
<span style="font-size: 30px;font-weight: bold"> Code Block 1: Dataset Initialization & Manifest Loading </span>

<span style="font-size: 22px;">
We import core computational and audio processing libraries. The dataset contains train and evaluation data for 6 classes (25 samples per class).

* **Directory Part** `/kaggle/input/datasets/ndeepanbalaji/ie-643-curated-dataset`.
* **Manifest Structure**: `manifest.csv` containing metadata attributes (e.g., file paths, species labels, split assignments).</span>

In [9]:
import os
import pandas as pd
import numpy as np
from pydub import AudioSegment
import torchaudio

DATA_DIR = "/kaggle/input/datasets/ndeepanbalaji/ie-643-curated-dataset"
manifest = pd.read_csv(os.path.join(DATA_DIR, "manifest.csv"))
print(manifest.shape)


(300, 5)


 <span style="font-size: 30px;font-weight: bold"> Code Block 2: Audio Resampling</span>
 
 <span style="font-size: 22px;"> 
Pretrained neural audio architectures require a uniform sampling rate ($16\text{ kHz}$) and single-channel (mono) input tensors.
$$x_{16\text{kHz}} = \text{Resample}_{f_{\text{orig}} \to 16000}(x_{\text{mono}})$$</span>


In [10]:
TARGET_SR = 16000

def load_and_resample(filepath, target_sr=TARGET_SR):
    
    waveform, orig_sr = torchaudio.load(filepath)
    if waveform.shape[0] > 1:
        waveform = waveform.mean(dim=0, keepdim=True)
    if orig_sr != target_sr:
        resampler = torchaudio.transforms.Resample(orig_freq=orig_sr, new_freq=target_sr)
        waveform = resampler(waveform)
    
    return waveform.squeeze(0).numpy()

 <span style="font-size: 30px;font-weight: bold"> Code Block 3: Pretrained Audio Model Setup & Embedding Extraction</span>

 <span style="font-size: 22px;">
We have used these 2-models for a baseline implementation. We will expand it to more models.

  * `hubert_base`: `facebook/hubert-base-ls960`
  * `wavlm_base`: `microsoft/wavlm-base`

* **Representation Pooling**:
  For a given resampled 1D raw waveform $x_{16\text{kHz}}$, the HuggingFace model outputs frame-level hidden representations $H \in \mathbb{R}^{B \times T \times D}$, where $B=1$ is the batch size, $T$ is the number of temporal frame windows, and $D = 768$ is the embedding dimension. We collapse them into a fixed-size sequence-level feature vector $z \in \mathbb{R}^{D}$ using mean-pooling.
  
  $$z = \frac{1}{T} \sum_{t=1}^{T} H[:, t, :] \in \mathbb{R}^{768}$$</span>

In [11]:
import torch
from transformers import AutoFeatureExtractor, AutoModel

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

def load_hf_model_and_processor(model_name, device=DEVICE):
    processor = AutoFeatureExtractor.from_pretrained(model_name)
    model = AutoModel.from_pretrained(model_name)
    model.eval().to(device)
    return model, processor

def extract_embedding(waveform, model, processor, device=DEVICE):
    inputs = processor(waveform, sampling_rate=TARGET_SR, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        hidden_states = model(**inputs).last_hidden_state
    return hidden_states.mean(dim=1).squeeze(0).cpu().numpy()

def get_embedding_for_clip(filepath, model, processor, device=DEVICE):
    wav = load_and_resample(filepath)
    return extract_embedding(wav, model, processor, device)

MODEL_REGISTRY = {
    "hubert_base":  {"type": "hf", "name": "facebook/hubert-base-ls960"},
    "wavlm_base":   {"type": "hf", "name": "microsoft/wavlm-base"},
}

def extract_all_embeddings(model_key, model_info, manifest_subset):
    
    model, processor = load_hf_model_and_processor(model_info["name"])
    embeddings = []
    labels = []
    for _, row in manifest_subset.iterrows():
        filepath = os.path.join(DATA_DIR, row.split, row.species, row.filename)
        emb = get_embedding_for_clip(filepath, model, processor)
        embeddings.append(emb)
        labels.append(row.species)
    
    del model 
    torch.cuda.empty_cache()
    
    return np.stack(embeddings), np.array(labels)

<span style="font-size: 30px;font-weight: bold"> Code Block 5: 25-Shot Processing </span>

<span style="font-size: 22px;">
We apply the extraction pipeline to the 25-shot per-class training pool (`train_pool`), consisting of 150 total bioacoustic audio clips across 6 classes. The extracted dimensions are matrices of the shape: 

  * `hubert_base`:  $(150, 768)$
  * `wavlm_base`: $(150, 768)$
    
We also save these extracted feature matrices and label vectors to `/kaggle/working/embeddings_25shot.npz` to avoid re-computation during scoring experiments.</span>

In [12]:
train_subset = manifest[manifest.split == "train_pool"].reset_index(drop=True)
all_embeddings = {}
all_labels = {}

for model_key, model_info in MODEL_REGISTRY.items():
    print(f"\nExtracting: {model_key}")
    emb, lab = extract_all_embeddings(model_key, model_info, train_subset)
    if emb is not None:
        all_embeddings[model_key] = emb
        all_labels[model_key] = lab
        print(f"  -> embeddings shape: {emb.shape}")

np.savez("/kaggle/working/embeddings_25shot.npz",
         **{f"{k}_emb": v for k, v in all_embeddings.items()},
         **{f"{k}_lab": v for k, v in all_labels.items()})
print("\nSaved embeddings to /kaggle/working/embeddings_25shot.npz")


Extracting: hubert_base


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

  -> embeddings shape: (150, 768)

Extracting: wavlm_base


Loading weights:   0%|          | 0/248 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/nn/functional.py:6371: UserWarning: Support for mismatched key_padding_mask and attn_mask is deprecated. Use same type for both instead.
  key_padding_mask = _canonical_mask(


  -> embeddings shape: (150, 768)

Saved embeddings to /kaggle/working/embeddings_25shot.npz


 <span style="font-size: 30px;font-weight: bold"> Code Block 6: NTK Label-Feature Correlation (LFC) Transferability Scoring </span>

 <span style="font-size: 22px">
     
Given $N$ samples with extracted feature vectors $\{z_1, z_2, \dots, z_N\} \in \mathbb{R}^{768}$ and species labels $\{y_1, y_2, \dots, y_N\}$:

1. Feature Cosine Similarity Matrix ($S \in \mathbb{R}^{N \times N}$):
   $$S_{ij} = \frac{z_i \cdot z_j}{\Vert{}z_i\Vert{}_2 \Vert{}z_j\Vert{}_2}$$

2. Label Binary Agreement Matrix ($A \in \mathbb{R}^{N \times N}$):
   $$A_{ij} = \begin{cases} 1, & \text{if } y_i = y_j \\ 0, & \text{if } y_i \neq y_j \end{cases}$$

3. Upper-Triangular Flattening
   To avoid duplicate pairs and self-correlations, extract indices $i < j$:
   $$\mathbf{s} = \{ S_{ij} \mid 1 \le i < j \le N \}, \quad \mathbf{a} = \{ A_{ij} \mid 1 \le i < j \le N \}$$

4. Spearman Rank Correlation ($\text{LFC}$):
   $$\text{LFC} = \rho_{\text{Spearman}}(\mathbf{s}, \mathbf{a}) = 1 - \frac{6 \sum_{k=1}^{M} d_k^2}{M(M^2 - 1)}$$
   where $M = \frac{N(N-1)}{2}$ is the number of unique sample pairs, and $d_k$ is the difference between the ranks of $\mathbf{s}_k$ and $\mathbf{a}_k$.

#### Benchmark Results (25-Shot Pool)
* **`hubert_base` LFC**: $0.1157$
* **`wavlm_base` LFC**: $0.0770$

**Predicted Model Ranking**: `hubert_base` $>$ `wavlm_base`. The higher LFC score indicates that `hubert_base` forms representation spaces better aligned with species label boundaries prior to fine-tuning. </span>

In [14]:
from scipy.stats import spearmanr
from sklearn.metrics.pairwise import cosine_similarity

def label_feature_correlation(embeddings, labels):
    n = len(embeddings)
    emb_sim = cosine_similarity(embeddings)            
    labels_arr = np.array(labels)                      
    label_agree = (labels_arr[:, None] == labels_arr[None, :]).astype(float)
    
    iu = np.triu_indices(n, k=1)
    emb_sim_flat = emb_sim[iu]
    label_agree_flat = label_agree[iu]
    
    corr, _ = spearmanr(emb_sim_flat, label_agree_flat)
    return corr

scores = {}
for model_key in all_embeddings:
    score = label_feature_correlation(all_embeddings[model_key], all_labels[model_key])
    scores[model_key] = score
    print(f"{model_key}: Label-Feature Correlation = {score:.4f}")

# print("\nFinal ranking (higher = better predicted transferability):")
# for k, v in sorted(scores.items(), key=lambda x: -x[1]):
#     print(f"  {k}: {v:.4f}")

hubert_base: Label-Feature Correlation = 0.1157
wavlm_base: Label-Feature Correlation = 0.0770
